### Seeing the Problem (Skewed Data)


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skew

# 🏥 Hospital bill data (right-skewed)
bills = pd.DataFrame({
    'patient': ['A','B','C','D','E','F','G','H'],
    'bill_amount': [500,800,1200,1500,3000,8000,50000,500000]
})

# print(bills['bill_amount'].skew())

# 📊 Check skewness
skewness = skew(bills['bill_amount'])
print(f"Skewness: {skewness:.2f}")
print(f"Mean: ₹{bills['bill_amount'].mean():,.0f}")
print(f"Median: ₹{bills['bill_amount'].median():,.0f}")
print(f"\n⚠️ Mean >> Median = RIGHT SKEWED!")
print(f"💡 PowerTransformer se fix karenge!")

Skewness: 2.23
Mean: ₹70,625
Median: ₹2,250

⚠️ Mean >> Median = RIGHT SKEWED!
💡 PowerTransformer se fix karenge!


### Box-Cox Transform


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer
from scipy.stats import skew

# 💰 Salary data (strictly positive, right-skewed)
salaries = np.array([[20000],[25000],[30000],[45000],
                     [60000],[100000],[500000],[2000000]])

# 📦 Box-Cox Transform
pt_boxcox = PowerTransformer(method='box-cox')
salaries_transformed = pt_boxcox.fit_transform(salaries)

print("📊 Before Box-Cox:")
print(f"  Skewness: {skew(salaries.flatten()):.2f}")
print(f"  Values: {salaries.flatten()}")

print(f"\n📦 After Box-Cox:")
print(f"  Skewness: {skew(salaries_transformed.flatten()):.2f}")
print(f"  Values: {np.round(salaries_transformed.flatten(),2)}")

print(f"\n🔑 Learned Lambda: {pt_boxcox.lambdas_[0]:.4f}")

df1 = pd.DataFrame(salaries, columns=['salaries'])
print(df1)

df2 = pd.DataFrame(salaries_transformed, columns=['salaries_transformed'])
print(df2)

📊 Before Box-Cox:
  Skewness: 2.05
  Values: [  20000   25000   30000   45000   60000  100000  500000 2000000]

📦 After Box-Cox:
  Skewness: 0.34
  Values: [-1.34 -1.01 -0.76 -0.28  0.01  0.44  1.28  1.64]

🔑 Learned Lambda: -0.4654
   salaries
0     20000
1     25000
2     30000
3     45000
4     60000
5    100000
6    500000
7   2000000
   salaries_transformed
0             -1.339345
1             -1.006097
2             -0.758368
3             -0.277304
4              0.012957
5              0.441651
6              1.283722
7              1.642784


### Yeo-Johnson Transform


In [ ]:
import numpy as np
from sklearn.preprocessing import PowerTransformer
from scipy.stats import skew

# 🌡️ Temperature data (has negatives & zeros!)
temps = np.array([[-10],[-5],[0],[5],[15],
                  [25],[35],[45]])

# 🔮 Yeo-Johnson Transform (default!)
pt_yj = PowerTransformer(method='yeo-johnson')
temps_transformed = pt_yj.fit_transform(temps)

print("🌡️ Before Yeo-Johnson:")
print(f"  Values: {temps.flatten()}")
print(f"  Skewness: {skew(temps.flatten()):.2f}")

print(f"\n🔮 After Yeo-Johnson:")
print(f"  Values: {np.round(temps_transformed.flatten(),2)}")
print(f"  Skewness: {skew(temps_transformed.flatten()):.2f}")

print(f"\n🔑 Lambda: {pt_yj.lambdas_[0]:.4f}")
print(f"✅ Negatives & zeros handled perfectly!")

🌡️ Before Yeo-Johnson:
  Values: [-10  -5   0   5  15  25  35  45]
  Skewness: 0.36

🔮 After Yeo-Johnson:
  Values: [-1.65 -1.05 -0.55 -0.23  0.26  0.69  1.08  1.46]
  Skewness: -0.14

🔑 Lambda: 0.7991
✅ Negatives & zeros handled perfectly!


### Comparison


In [ ]:
import numpy as np
from sklearn.preprocessing import PowerTransformer
from scipy.stats import skew

# 💰 Positive-only data (both work here!)
data = np.array([[100],[200],[500],[1000],
                 [5000],[10000],[50000],[200000]])

# 📦 Box-Cox
bc = PowerTransformer(method='box-cox')
data_bc = bc.fit_transform(data)

# 🔮 Yeo-Johnson
yj = PowerTransformer(method='yeo-johnson')
data_yj = yj.fit_transform(data)

print(f"Original Skewness:    {skew(data.flatten()):.4f}")
print(f"Box-Cox Skewness:     {skew(data_bc.flatten()):.4f}")
print(f"Yeo-Johnson Skewness: {skew(data_yj.flatten()):.4f}")

print(f"\nBox-Cox λ:     {bc.lambdas_[0]:.4f}")
print(f"Yeo-Johnson λ: {yj.lambdas_[0]:.4f}")

print(f"\n💡 Positive data pe dono similar results dete hain!")

Original Skewness:    2.0341
Box-Cox Skewness:     0.0839
Yeo-Johnson Skewness: 0.0846

Box-Cox λ:     -0.0769
Yeo-Johnson λ: -0.0775

💡 Positive data pe dono similar results dete hain!


### ColumnTransformer


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder

# 🏥 Patient data (mixed types!)
patients = pd.DataFrame({
    'age': [25,45,35,60,28,50],
    'gender': ['M','F','M','F','M','F'],
    'bill': [500,8000,1200,50000,800,25000],
    'city': ['Surat','Ahmedabad','Surat',
             'Mumbai','Ahmedabad','Mumbai']
})

print("📋 Original Data:")
print(patients.to_string(index=False))

# 🏗️ ColumnTransformer — alag columns, alag treatment!
ct = ColumnTransformer(
      transformers=[
      ('scale_age', StandardScaler(), ['age']),
      ('power_bill', PowerTransformer(), ['bill']),
      ('encode_cat', OneHotEncoder(sparse_output=False), ['gender','city'])
  ], remainder='drop')

result = ct.fit_transform(patients)

# Get feature names
feature_names = ct.get_feature_names_out()
print(f"\n🏗️ After ColumnTransformer:")
print(f"Feature names: {list(feature_names)}")
print(f"Shape: {result.shape}")

print(f"\nTransformed:")
df = pd.DataFrame(result, columns=feature_names)
df


📋 Original Data:
 age gender  bill      city
  25      M   500     Surat
  45      F  8000 Ahmedabad
  35      M  1200     Surat
  60      F 50000    Mumbai
  28      M   800 Ahmedabad
  50      F 25000    Mumbai

🏗️ After ColumnTransformer:
Feature names: ['scale_age__age', 'power_bill__bill', 'encode_cat__gender_F', 'encode_cat__gender_M', 'encode_cat__city_Ahmedabad', 'encode_cat__city_Mumbai', 'encode_cat__city_Surat']
Shape: (6, 7)

Transformed:


,scale_age__age,power_bill__bill,encode_cat__gender_F,encode_cat__gender_M,encode_cat__city_Ahmedabad,encode_cat__city_Mumbai,encode_cat__city_Surat
0,-1.253442,-1.256837,0.0,1.0,0.0,0.0,1.0
1,0.363903,0.459684,1.0,0.0,1.0,0.0,0.0
2,-0.444770,-0.661527,0.0,1.0,0.0,0.0,1.0
3,1.576911,1.353987,1.0,0.0,0.0,1.0,0.0
4,-1.010840,-0.930678,0.0,1.0,1.0,0.0,0.0
5,0.768239,1.035371,1.0,0.0,0.0,1.0,0.0


### ColumnTransformer + Pipeline


In [ ]:
|from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

# 🏗️ Step 1: Define ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['age']),
    ('power', PowerTransformer(), ['bill']),
    ('cat', OneHotEncoder(), ['gender','city'])
])

transformed = preprocessor.fit_transform(patients)

df_transformed = pd.DataFrame(transformed, columns=preprocessor.get_feature_names_out())
print(df_transformed.to_string())

# 🔗 Step 2: Create Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# 🚀 Step 3: Fit & Predict (one line!)
# pipeline.fit(X_train, y_train)
# predictions = pipeline.predict(X_test)

print("\n✅ Pipeline ready!")
print("💡 One .fit() call → preprocess + train!")
print("💡 One .predict() call → preprocess + predict!")

   num__age  power__bill  cat__gender_F  cat__gender_M  cat__city_Ahmedabad  cat__city_Mumbai  cat__city_Surat
0 -1.253442    -1.256837            0.0            1.0                  0.0               0.0              1.0
1  0.363903     0.459684            1.0            0.0                  1.0               0.0              0.0
2 -0.444770    -0.661527            0.0            1.0                  0.0               0.0              1.0
3  1.576911     1.353987            1.0            0.0                  0.0               1.0              0.0
4 -1.010840    -0.930678            0.0            1.0                  1.0               0.0              0.0
5  0.768239     1.035371            1.0            0.0                  0.0               1.0              0.0

✅ Pipeline ready!
💡 One .fit() call → preprocess + train!
💡 One .predict() call → preprocess + predict!
